In [2]:
!pip install medmnist torch torchvision scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.3 MB/s eta 0:00:00


In [6]:
import copy
import os
from dataclasses import dataclass
from typing import Dict, Tuple

import medmnist
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from medmnist import INFO
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm import tqdm


# -----------------------------
# Config
# -----------------------------
@dataclass
class Config:
    data_flag: str = "octmnist"
    batch_size: int = 128
    num_workers: int = 2
    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_classes: int = 4
    image_size: int = 28
    pretrained: bool = False
    experiment_name: str = "resnet18_28_scratch"
    save_dir: str = "./checkpoints"


# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------
# Dataset / Dataloader
# -----------------------------
def get_transforms(image_size: int, pretrained: bool):
    # OCTMNIST images are grayscale, ResNet expects 3 channels.
    # So we convert grayscale to 3 identical channels.
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]

    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    test_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return train_transform, test_transform


def get_dataloaders(cfg: Config):
    info = INFO[cfg.data_flag]
    DataClass = getattr(medmnist, info["python_class"])

    train_transform, test_transform = get_transforms(cfg.image_size, cfg.pretrained)

    train_dataset = DataClass(split="train", transform=train_transform, download=True)
    val_dataset = DataClass(split="val", transform=test_transform, download=True)
    test_dataset = DataClass(split="test", transform=test_transform, download=True)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader


# -----------------------------
# Model
# -----------------------------
def build_resnet18(cfg: Config) -> nn.Module:
    if cfg.pretrained:
        weights = models.ResNet18_Weights.DEFAULT
    else:
        weights = None

    model = models.resnet18(weights=weights)

    # If using 28x28, the default ImageNet stem is a bit aggressive.
    # We make it CIFAR-style:
    if cfg.image_size == 28:
        model.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        model.maxpool = nn.Identity()

    model.fc = nn.Linear(model.fc.in_features, cfg.num_classes)
    return model


# -----------------------------
# Metrics
# -----------------------------
def compute_multiclass_auc(y_true: np.ndarray, y_prob: np.ndarray, num_classes: int) -> float:
    y_true_onehot = np.eye(num_classes)[y_true]
    try:
        auc = roc_auc_score(
            y_true_onehot,
            y_prob,
            multi_class="ovr",
            average="macro",
        )
    except ValueError:
        auc = float("nan")
    return auc


# -----------------------------
# Train / Eval
# -----------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device)
        labels = labels.squeeze().long().to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_probs.append(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_auc = compute_multiclass_auc(all_labels, all_probs, num_classes)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "auc": epoch_auc,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
    }


# -----------------------------
# Full training loop
# -----------------------------
def fit(cfg: Config):
    os.makedirs(cfg.save_dir, exist_ok=True)
    set_seed(42)

    train_loader, val_loader, test_loader = get_dataloaders(cfg)

    model = build_resnet18(cfg).to(cfg.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    best_val_acc = -1.0
    best_model_state = None
    history = []

    print(f"\n=== Running {cfg.experiment_name} ===")
    print(cfg)

    for epoch in range(cfg.epochs):
        print(f"\nEpoch [{epoch+1}/{cfg.epochs}]")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, cfg.device
        )
        val_metrics = evaluate(
            model, val_loader, criterion, cfg.device, cfg.num_classes
        )

        scheduler.step()

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['acc']:.4f} | "
            f"Val AUC: {val_metrics['auc']:.4f}"
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_auc": val_metrics["auc"],
        })

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_model_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_state)

    ckpt_path = os.path.join(cfg.save_dir, f"{cfg.experiment_name}.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
        "best_val_acc": best_val_acc,
        "history": history,
    }, ckpt_path)

    print(f"\nBest model saved to: {ckpt_path}")

    val_metrics = evaluate(model, val_loader, criterion, cfg.device, cfg.num_classes)
    test_metrics = evaluate(model, test_loader, criterion, cfg.device, cfg.num_classes)

    print("\n=== Final Results ===")
    print(
        f"Validation -> Loss: {val_metrics['loss']:.4f}, "
        f"Acc: {val_metrics['acc']:.4f}, AUC: {val_metrics['auc']:.4f}"
    )
    print(
        f"Test       -> Loss: {test_metrics['loss']:.4f}, "
        f"Acc: {test_metrics['acc']:.4f}, AUC: {test_metrics['auc']:.4f}"
    )

    return model, history, val_metrics, test_metrics


# -----------------------------
# Run experiments
# -----------------------------
if __name__ == "__main__":
    import numpy as np
    import os

    os.makedirs("./outputs", exist_ok=True)

    experiments = [
        Config(
            image_size=28,
            pretrained=False,
            experiment_name="resnet18_28_scratch",
            epochs=5,
            batch_size=128,
        ),
        Config(
            image_size=224,
            pretrained=False,
            experiment_name="resnet18_224_scratch",
            epochs=5,
            batch_size=64,
        ),
        # Config(
        #     image_size=224,
        #     pretrained=True,
        #     experiment_name="resnet18_224_pretrained",
        #     epochs=3,
        #     batch_size=64,
        # ),
    ]

    results = {}

    for cfg in experiments:
        model, history, val_metrics, test_metrics = fit(cfg)

        # 🔥 CONFORMAL İÇİN SAVE
        output_path = f"./outputs/{cfg.experiment_name}_outputs.npz"

        np.savez(
            output_path,
            val_probs=val_metrics["probs"],
            val_labels=val_metrics["labels"],
            test_probs=test_metrics["probs"],
            test_labels=test_metrics["labels"],
        )

        print(f"Saved outputs to {output_path}")

        results[cfg.experiment_name] = {
            "val_acc": val_metrics["acc"],
            "val_auc": val_metrics["auc"],
            "test_acc": test_metrics["acc"],
            "test_auc": test_metrics["auc"],
        }

    print("\n=== Summary ===")
    for exp_name, metrics in results.items():
        print(
            f"{exp_name} | "
            f"Val Acc: {metrics['val_acc']:.4f} | Val AUC: {metrics['val_auc']:.4f} | "
            f"Test Acc: {metrics['test_acc']:.4f} | Test AUC: {metrics['test_auc']:.4f}"
        )


=== Running resnet18_28_scratch ===
Config(data_flag='octmnist', batch_size=128, num_workers=2, epochs=5, lr=0.001, weight_decay=0.0001, device='cuda', num_classes=4, image_size=28, pretrained=False, experiment_name='resnet18_28_scratch', save_dir='./checkpoints')

Epoch [1/5]


Train Loss: 0.4198 | Train Acc: 0.8551 | Val Loss: 0.3185 | Val Acc: 0.8942 | Val AUC: 0.9592

Epoch [2/5]


Train Loss: 0.2943 | Train Acc: 0.8990 | Val Loss: 0.4038 | Val Acc: 0.8630 | Val AUC: 0.9444

Epoch [3/5]


Train Loss: 0.2553 | Train Acc: 0.9123 | Val Loss: 0.3082 | Val Acc: 0.8947 | Val AUC: 0.9665

Epoch [4/5]


Train Loss: 0.2162 | Train Acc: 0.9262 | Val Loss: 0.5149 | Val Acc: 0.8389 | Val AUC: 0.9430

Epoch [5/5]


...

Train Loss: 0.3369 | Train Acc: 0.8846 | Val Loss: 0.3693 | Val Acc: 0.8791 | Val AUC: 0.9538

Epoch [3/5]


Train Loss: 0.2911 | Train Acc: 0.9004 | Val Loss: 0.3548 | Val Acc: 0.8804 | Val AUC: 0.9566

Epoch [4/5]


Train Loss: 0.2509 | Train Acc: 0.9139 | Val Loss: 0.3154 | Val Acc: 0.8894 | Val AUC: 0.9646

Epoch [5/5]


Train Loss: 0.2068 | Train Acc: 0.9294 | Val Loss: 0.2260 | Val Acc: 0.9224 | Val AUC: 0.9772

Best model saved to: ./checkpoints/resnet18_224_scratch.pt



=== Final Results ===
Validation -> Loss: 0.2260, Acc: 0.9224, AUC: 0.9772
Test       -> Loss: 0.5498, Acc: 0.7880, AUC: 0.9745
Saved outputs to ./outputs/resnet18_224_scratch_outputs.npz

=== Summary ===
resnet18_28_scratch | Val Acc: 0.9294 | Val AUC: 0.9796 | Test Acc: 0.7290 | Test AUC: 0.9526
resnet18_224_scratch | Val Acc: 0.9224 | Val AUC: 0.9772 | Test Acc: 0.7880 | Test AUC: 0.9745


In [ ]:
import os
from dataclasses import dataclass

import medmnist
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from medmnist import INFO
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm import tqdm


# -----------------------------
# Config
# -----------------------------
@dataclass
class Config:
    data_flag: str = "tissuemnist"
    batch_size: int = 128
    num_workers: int = 2
    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_classes: int = 8
    image_size: int = 28
    experiment_name: str = "resnet18_28_tissuemnist_scratch"
    save_dir: str = "./checkpoints"
    output_dir: str = "./outputs"


# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Optional deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------
# Dataset / Dataloader
# -----------------------------
def get_transforms(image_size: int):
    # OCTMNIST images are grayscale, ResNet expects 3 channels.
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]

    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    test_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return train_transform, test_transform


def get_dataloaders(cfg: Config):
    info = INFO[cfg.data_flag]
    DataClass = getattr(medmnist, info["python_class"])

    train_transform, test_transform = get_transforms(cfg.image_size)

    train_dataset = DataClass(split="train", transform=train_transform, download=True)
    val_dataset = DataClass(split="val", transform=test_transform, download=True)
    test_dataset = DataClass(split="test", transform=test_transform, download=True)

    pin_memory = cfg.device.startswith("cuda")

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader, test_loader


# -----------------------------
# Model
# -----------------------------
def build_resnet18(cfg: Config) -> nn.Module:
    model = models.resnet18(weights=None)

    # If using 28x28, default ImageNet stem is too aggressive.
    if cfg.image_size == 28:
        model.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        model.maxpool = nn.Identity()

    model.fc = nn.Linear(model.fc.in_features, cfg.num_classes)
    return model


# -----------------------------
# Metrics
# -----------------------------
def compute_multiclass_auc(y_true: np.ndarray, y_prob: np.ndarray, num_classes: int) -> float:
    y_true_onehot = np.eye(num_classes)[y_true]
    try:
        auc = roc_auc_score(
            y_true_onehot,
            y_prob,
            multi_class="ovr",
            average="macro",
        )
    except ValueError:
        auc = float("nan")
    return auc


# -----------------------------
# Train / Eval
# -----------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.view(-1).long().to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device)
        labels = labels.view(-1).long().to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_probs.append(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_auc = compute_multiclass_auc(all_labels, all_probs, num_classes)

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "auc": epoch_auc,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
    }


# -----------------------------
# Full training loop
# -----------------------------
def fit(cfg: Config):
    os.makedirs(cfg.save_dir, exist_ok=True)
    os.makedirs(cfg.output_dir, exist_ok=True)

    set_seed(42)

    train_loader, val_loader, test_loader = get_dataloaders(cfg)

    model = build_resnet18(cfg).to(cfg.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    history = []

    print(f"\n=== Running {cfg.experiment_name} ===")
    print(cfg)

    for epoch in range(cfg.epochs):
        print(f"\nEpoch [{epoch + 1}/{cfg.epochs}]")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, cfg.device
        )
        val_metrics = evaluate(
            model, val_loader, criterion, cfg.device, cfg.num_classes
        )

        scheduler.step()

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['acc']:.4f} | "
            f"Val AUC: {val_metrics['auc']:.4f}"
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_auc": val_metrics["auc"],
        })

    # Save final model (not best-val-selected model)
    ckpt_path = os.path.join(cfg.save_dir, f"{cfg.experiment_name}.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": cfg.__dict__,
        "history": history,
    }, ckpt_path)

    print(f"\nFinal model saved to: {ckpt_path}")

    val_metrics = evaluate(model, val_loader, criterion, cfg.device, cfg.num_classes)
    test_metrics = evaluate(model, test_loader, criterion, cfg.device, cfg.num_classes)

    print("\n=== Final Results ===")
    print(
        f"Validation -> Loss: {val_metrics['loss']:.4f}, "
        f"Acc: {val_metrics['acc']:.4f}, AUC: {val_metrics['auc']:.4f}"
    )
    print(
        f"Test       -> Loss: {test_metrics['loss']:.4f}, "
        f"Acc: {test_metrics['acc']:.4f}, AUC: {test_metrics['auc']:.4f}"
    )

    # Save outputs for conformal
    output_path = os.path.join(cfg.output_dir, f"{cfg.experiment_name}_outputs.npz")
    np.savez(
        output_path,
        val_probs=val_metrics["probs"],
        val_labels=val_metrics["labels"],
        test_probs=test_metrics["probs"],
        test_labels=test_metrics["labels"],
    )

    print(f"Saved outputs to: {output_path}")

    return model, history, val_metrics, test_metrics


# -----------------------------
# Run experiments
# -----------------------------
if __name__ == "__main__":
    experiments = [
        Config(
            data_flag="tissuemnist",
            num_classes=8,
            image_size=28,
            experiment_name="resnet18_28_tissuemnist_scratch",
            epochs=10,
            batch_size=128,
        ),
        Config(
            data_flag="tissuemnist",
            num_classes=8,
            image_size=224,
            experiment_name="resnet18_224_tissuemnist_scratch",
            epochs=10,
            batch_size=64,
        ),
    ]

    results = {}

    for cfg in experiments:
        model, history, val_metrics, test_metrics = fit(cfg)

        results[cfg.experiment_name] = {
            "val_acc": val_metrics["acc"],
            "val_auc": val_metrics["auc"],
            "test_acc": test_metrics["acc"],
            "test_auc": test_metrics["auc"],
        }

    print("\n=== Summary ===")
    for exp_name, metrics in results.items():
        print(
            f"{exp_name} | "
            f"Val Acc: {metrics['val_acc']:.4f} | Val AUC: {metrics['val_auc']:.4f} | "
            f"Test Acc: {metrics['test_acc']:.4f} | Test AUC: {metrics['test_auc']:.4f}"
        )


=== Running resnet18_28_scratch ===
Config(data_flag='octmnist', batch_size=128, num_workers=2, epochs=10, lr=0.001, weight_decay=0.0001, device='cuda', num_classes=4, image_size=28, experiment_name='resnet18_28_scratch', save_dir='./checkpoints', output_dir='./outputs')

Epoch [1/10]


Train Loss: 0.4222 | Train Acc: 0.8541 | Val Loss: 0.3345 | Val Acc: 0.8938 | Val AUC: 0.9605

Epoch [2/10]


Train Loss: 0.2986 | Train Acc: 0.8976 | Val Loss: 0.3267 | Val Acc: 0.8911 | Val AUC: 0.9563

Epoch [3/10]


Train Loss: 0.2668 | Train Acc: 0.9077 | Val Loss: 0.4725 | Val Acc: 0.8414 | Val AUC: 0.9481

Epoch [4/10]


Train Loss: 0.2460 | Train Acc: 0.9163 | Val Loss: 0.7664 | Val Acc: 0.7843 | Val AUC: 0.9197

Epoch [5/10]


...

Train Loss: 0.2105 | Train Acc: 0.9277 | Val Loss: 0.2252 | Val Acc: 0.9270 | Val AUC: 0.9762

Epoch [8/10]


Train Loss: 0.1787 | Train Acc: 0.9385 | Val Loss: 0.2188 | Val Acc: 0.9276 | Val AUC: 0.9794

Epoch [9/10]


Train Loss: 0.1453 | Train Acc: 0.9505 | Val Loss: 0.2042 | Val Acc: 0.9340 | Val AUC: 0.9808

Epoch [10/10]


Train Loss: 0.1154 | Train Acc: 0.9618 | Val Loss: 0.2073 | Val Acc: 0.9349 | Val AUC: 0.9810

Final model saved to: ./checkpoints/resnet18_224_scratch.pt



=== Final Results ===
Validation -> Loss: 0.2073, Acc: 0.9349, AUC: 0.9810
Test       -> Loss: 0.7422, Acc: 0.7730, AUC: 0.9620
Saved outputs to: ./outputs/resnet18_224_scratch_outputs.npz

=== Summary ===
resnet18_28_scratch | Val Acc: 0.9346 | Val AUC: 0.9816 | Test Acc: 0.7460 | Test AUC: 0.9530
resnet18_224_scratch | Val Acc: 0.9349 | Val AUC: 0.9810 | Test Acc: 0.7730 | Test AUC: 0.9620
